# Project 2 — Molecular Descriptor Dataset and QSAR Regression

This notebook builds a complete descriptor-based QSAR workflow using RDKit and scikit-learn.

The practice goal is to move from individual molecule analysis in Project 1 to a real machine-learning dataset:

```text
SMILES → RDKit molecule → molecular descriptors → feature matrix → regression model → property prediction
```

Target property:

- Measured log solubility, `logS`, from the ESOL / Delaney dataset.
- Lower logS usually means poorer aqueous solubility.
- Higher logS usually means better aqueous solubility.

This notebook is intentionally detailed for GitHub portfolio use. It demonstrates chemistry, cheminformatics and machine-learning workflow skills.

## Learning objectives

By the end of this notebook, you should be able to:

1. Load a public molecular property dataset.
2. Validate and canonicalise SMILES.
3. Calculate RDKit molecular descriptors.
4. Build a clean descriptor matrix.
5. Train and compare multiple QSAR regression models.
6. Use train/test metrics and cross-validation.
7. Diagnose errors using residuals and molecule-level inspection.
8. Interpret descriptor importance chemically.
9. Predict logS for new SMILES.
10. Save processed data and model result tables for later projects.

## Notebook chapters

| Chapter | Topic |
|---|---|
| 1 | Imports and settings |
| 2 | Load ESOL / Delaney dataset |
| 3 | SMILES validation and canonicalisation |
| 4 | RDKit descriptor calculation |
| 5 | Lipinski and descriptor sanity checks |
| 6 | Exploratory descriptor analysis |
| 7 | Feature matrix preparation |
| 8 | Train/test split |
| 9 | Model benchmarking |
| 10 | Cross-validation |
| 11 | Final model diagnostics |
| 12 | Molecule-level error analysis |
| 13 | Feature interpretation |
| 14 | Predict new SMILES |
| 15 | Single-molecule report helper |
| 16 | Bonus miscibility heuristic |
| 17 | Save outputs |

## Environment note

Recommended conda setup:

```bash
conda install -c conda-forge rdkit scikit-learn pandas numpy matplotlib joblib
```

If running in Kaggle or Colab and RDKit is missing, install RDKit in a separate setup cell before running this notebook.

In [ ]:
# Optional setup for Colab/Kaggle if RDKit is not installed.
# Uncomment only when needed.
# !pip install rdkit

In [ ]:
# ============================================================
# Chapter 1 — Imports and settings
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, Draw
from rdkit.Chem.Draw import IPythonConsole

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
OUTPUT_DIR = Path("../outputs/project2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

print("Imports completed.")

# Chapter 2 — Load ESOL / Delaney dataset

The ESOL / Delaney dataset is a common QSAR regression dataset for aqueous solubility.

Expected important columns:

| Original column | Meaning |
|---|---|
| `Compound ID` | Molecule name / identifier |
| `smiles` | Molecular structure string |
| `measured log solubility in mols per litre` | Experimental logS target |

In [ ]:
# ============================================================
# Chapter 2 — Load ESOL / Delaney dataset
# ============================================================

ESOL_URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"

fallback_data = {
    "name": [
        "ethanol", "propanol", "butanol", "benzene", "toluene",
        "phenol", "acetic acid", "acetone", "ethyl acetate", "aspirin",
        "caffeine", "ibuprofen", "paracetamol", "limonene", "vanillin"
    ],
    "smiles": [
        "CCO", "CCCO", "CCCCO", "c1ccccc1", "Cc1ccccc1",
        "Oc1ccccc1", "CC(=O)O", "CC(=O)C", "CCOC(=O)C",
        "CC(=O)OC1=CC=CC=C1C(=O)O",
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
        "CC(=O)NC1=CC=C(O)C=C1",
        "CC1=CCC(CC1)C(=C)C",
        "COC1=C(C=CC(=C1)C=O)O"
    ],
    # Approximate practice targets only, used if public dataset loading fails.
    "target_logS": [
        0.0, -0.3, -0.8, -1.6, -2.2,
        -1.5, 1.2, 0.1, -0.7, -2.0,
        -0.6, -4.0, -0.8, -4.5, -1.2
    ],
}

try:
    raw_df = pd.read_csv(ESOL_URL)
    df = raw_df.rename(columns={
        "Compound ID": "name",
        "measured log solubility in mols per litre": "target_logS",
    })
    df = df[["name", "smiles", "target_logS"]].copy()
    data_source = "ESOL / Delaney public URL"
except Exception as exc:
    print("Could not load public ESOL dataset. Using embedded fallback dataset.")
    print("Reason:", exc)
    df = pd.DataFrame(fallback_data)
    data_source = "embedded fallback dataset"

df = df.dropna(subset=["smiles", "target_logS"]).reset_index(drop=True)
df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
df = df.dropna(subset=["target_logS"]).reset_index(drop=True)

print("Data source:", data_source)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Inspect target distribution.
print("Target column: target_logS")
display(df["target_logS"].describe())

plt.figure()
plt.hist(df["target_logS"], bins=30)
plt.xlabel("Measured logS")
plt.ylabel("Count")
plt.title("Distribution of measured log solubility")
plt.show()

# Chapter 3 — SMILES validation and canonicalisation

SMILES must be converted into RDKit `Mol` objects before descriptors can be calculated.

Important checks:

| Step | Purpose |
|---|---|
| Parse SMILES | Remove invalid molecules |
| Canonical SMILES | Standardise equivalent structures |
| Duplicate removal | Avoid repeated molecules leaking into train/test sets |
| Keep row index | Preserve molecule-level traceability |

In [ ]:
# ============================================================
# Chapter 3 — SMILES validation and canonicalisation
# ============================================================

def smiles_to_mol(smiles):
    # Convert a SMILES string into an RDKit Mol object.
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


def mol_to_canonical_smiles(mol):
    # Convert an RDKit Mol object to canonical SMILES.
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


df["mol"] = df["smiles"].apply(smiles_to_mol)
df["valid_mol"] = df["mol"].notna()

invalid_rows = df[~df["valid_mol"]].copy()
print("Invalid SMILES count:", len(invalid_rows))

if len(invalid_rows) > 0:
    display(invalid_rows[["name", "smiles"]].head())

df = df[df["valid_mol"]].copy().reset_index(drop=True)
df["canonical_smiles"] = df["mol"].apply(mol_to_canonical_smiles)

before_dedup = len(df)
df = df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
after_dedup = len(df)

print("Rows before duplicate removal:", before_dedup)
print("Rows after duplicate removal:", after_dedup)
display(df[["name", "smiles", "canonical_smiles", "target_logS"]].head())

In [ ]:
# Visual inspection of a small molecule sample.
sample_size = min(12, len(df))
sample_df = df.sample(sample_size, random_state=RANDOM_STATE)

mols = sample_df["mol"].tolist()
legends = sample_df["name"].astype(str).tolist()

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=4,
    subImgSize=(220, 180),
    legends=legends
)

display(img)

# Chapter 4 — RDKit descriptor calculation

Descriptors are numerical chemical features derived from structure.

This project uses interpretable 2D descriptors so the model remains chemically explainable.

| Descriptor | Meaning | Typical relevance to solubility |
|---|---|---|
| `MolWt` | Molecular weight | Larger molecules often dissolve less easily |
| `MolLogP` | Lipophilicity / hydrophobicity | Higher LogP often means lower water solubility |
| `TPSA` | Topological polar surface area | Higher polarity often increases water solubility up to a point |
| `HBD` | Hydrogen bond donors | Can improve water interaction |
| `HBA` | Hydrogen bond acceptors | Can improve water interaction |
| `RotatableBonds` | Flexibility | Affects conformation and physicochemical behaviour |
| `RingCount` | Number of rings | Often relates to rigidity and hydrophobicity |
| `AromaticRingCount` | Number of aromatic rings | Aromaticity often increases hydrophobic character |
| `FractionCSP3` | Fraction of sp3 carbons | Captures saturation and 3D character |
| `BertzCT` | Molecular complexity | Higher values suggest more complex structures |
| `LabuteASA` | Approximate surface area | Related to molecular exposure and size |

In [ ]:
# ============================================================
# Chapter 4 — RDKit descriptor calculation
# ============================================================

def calculate_descriptors(mol):
    # Calculate a compact descriptor panel for QSAR regression.
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Lipinski.HeavyAtomCount(mol),
        "RingCount": Lipinski.RingCount(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "AliphaticRingCount": rdMolDescriptors.CalcNumAliphaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "BalabanJ": Descriptors.BalabanJ(mol),
        "LabuteASA": rdMolDescriptors.CalcLabuteASA(mol),
        "NumHeteroatoms": Lipinski.NumHeteroatoms(mol),
        "NumValenceElectrons": Descriptors.NumValenceElectrons(mol),
        "NHOHCount": Lipinski.NHOHCount(mol),
        "NOCount": Lipinski.NOCount(mol),
    }


descriptor_df = pd.DataFrame([calculate_descriptors(mol) for mol in df["mol"]])

df_desc = pd.concat(
    [
        df[["name", "smiles", "canonical_smiles", "target_logS", "mol"]].reset_index(drop=True),
        descriptor_df.reset_index(drop=True),
    ],
    axis=1,
)

print("Descriptor dataset shape:", df_desc.shape)
display(df_desc.head())

# Chapter 5 — Lipinski and descriptor sanity checks

Lipinski Rule of Five is not a solubility model, but it is useful for quickly checking molecular size, hydrophobicity and hydrogen-bonding load.

In this notebook, Lipinski features are used as interpretable descriptor summaries rather than strict drug-discovery filters.

In [ ]:
# ============================================================
# Chapter 5 — Lipinski summary features
# ============================================================

def lipinski_score(row):
    # Return the number of basic Lipinski rules passed.
    rules = [
        row["MolWt"] <= 500,
        row["MolLogP"] <= 5,
        row["HBD"] <= 5,
        row["HBA"] <= 10,
    ]
    return int(sum(rules))


def lipinski_pass(row):
    # Return True if all four basic Lipinski rules are passed.
    return lipinski_score(row) == 4


df_desc["LipinskiScore"] = df_desc.apply(lipinski_score, axis=1)
df_desc["LipinskiPass"] = df_desc.apply(lipinski_pass, axis=1)

display(
    df_desc[
        ["name", "target_logS", "MolWt", "MolLogP", "TPSA", "HBD", "HBA", "LipinskiScore", "LipinskiPass"]
    ].head(10)
)

print("Lipinski score distribution:")
display(df_desc["LipinskiScore"].value_counts().sort_index())

In [ ]:
# Show high molecular weight examples for manual inspection.
outlier_view = df_desc.sort_values("MolWt", ascending=False)[
    ["name", "target_logS", "MolWt", "MolLogP", "TPSA", "HBD", "HBA", "canonical_smiles"]
].head(10)

display(outlier_view)

# Chapter 6 — Exploratory descriptor analysis

Before training ML models, inspect whether descriptors have reasonable distributions and whether they relate to the target.

In [ ]:
# ============================================================
# Chapter 6 — Descriptor summary statistics
# ============================================================

numeric_cols = df_desc.select_dtypes(include=[np.number]).columns.tolist()
display(df_desc[numeric_cols].describe().T)

In [ ]:
# Distribution of key descriptors.
key_descriptors = ["MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotatableBonds"]

for col in key_descriptors:
    plt.figure()
    plt.hist(df_desc[col], bins=30)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
# Descriptor-target scatter plots.
scatter_features = ["MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotatableBonds"]

for col in scatter_features:
    plt.figure()
    plt.scatter(df_desc[col], df_desc["target_logS"], alpha=0.6)
    plt.xlabel(col)
    plt.ylabel("Measured logS")
    plt.title(f"{col} vs measured logS")
    plt.show()

In [ ]:
# Correlation with target.
corr_series = (
    df_desc[numeric_cols]
    .corr(numeric_only=True)["target_logS"]
    .drop("target_logS")
    .sort_values()
)

display(corr_series.to_frame("correlation_with_logS"))

plt.figure(figsize=(8, 6))
plt.barh(corr_series.index, corr_series.values)
plt.xlabel("Correlation with measured logS")
plt.ylabel("Descriptor")
plt.title("Descriptor correlation with solubility")
plt.show()

In [ ]:
# Compact correlation heatmap for main descriptors.
heatmap_cols = ["target_logS", "MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotatableBonds", "RingCount", "FractionCSP3"]
corr = df_desc[heatmap_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Descriptor correlation heatmap")
plt.tight_layout()
plt.show()

display(corr)

# Chapter 7 — Feature matrix preparation

The machine-learning model needs:

```text
X = descriptor feature matrix
y = target logS values
```

Preprocessing steps:

1. Select descriptor columns.
2. Replace infinities with missing values.
3. Remove rows with missing descriptor values.
4. Remove zero-variance descriptors.
5. Preserve molecule indices for later error analysis.

In [ ]:
# ============================================================
# Chapter 7 — Build feature matrix X and target y
# ============================================================

descriptor_cols = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "HeavyAtomCount",
    "RingCount",
    "AromaticRingCount",
    "AliphaticRingCount",
    "FractionCSP3",
    "BertzCT",
    "BalabanJ",
    "LabuteASA",
    "NumHeteroatoms",
    "NumValenceElectrons",
    "NHOHCount",
    "NOCount",
    "LipinskiScore",
]

model_df = df_desc[["name", "canonical_smiles", "target_logS"] + descriptor_cols].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan)

before_dropna = len(model_df)
model_df = model_df.dropna(subset=descriptor_cols + ["target_logS"]).reset_index(drop=True)
after_dropna = len(model_df)

print("Rows before dropping missing values:", before_dropna)
print("Rows after dropping missing values:", after_dropna)

X_raw = model_df[descriptor_cols].copy()
y = model_df["target_logS"].copy()

print("Raw X shape:", X_raw.shape)
print("y shape:", y.shape)
display(X_raw.head())

In [ ]:
# Remove zero-variance descriptors.
selector = VarianceThreshold(threshold=0.0)
X_selected_array = selector.fit_transform(X_raw)

selected_feature_names = X_raw.columns[selector.get_support()].tolist()
removed_feature_names = X_raw.columns[~selector.get_support()].tolist()

X = pd.DataFrame(X_selected_array, columns=selected_feature_names, index=model_df.index)

print("Selected X shape:", X.shape)
print("Removed zero-variance features:", removed_feature_names)
display(X.head())

# Chapter 8 — Train/test split

The split keeps molecule indices so later we can map prediction errors back to molecule names and structures.

In [ ]:
# ============================================================
# Chapter 8 — Train/test split
# ============================================================

indices = model_df.index.to_numpy()

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y,
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Training target range:", (float(y_train.min()), float(y_train.max())))
print("Test target range:", (float(y_test.min()), float(y_test.max())))

# Chapter 9 — Model benchmarking

We compare several regression algorithms:

| Model | Why include it |
|---|---|
| Linear Regression | Simple baseline |
| Ridge | Regularised linear baseline |
| Lasso | Sparse linear model |
| ElasticNet | Mixed L1/L2 regularisation |
| Random Forest | Strong nonlinear tree baseline |
| Extra Trees | Strong ensemble with high randomisation |
| Gradient Boosting | Sequential tree ensemble |
| SVR RBF | Nonlinear kernel baseline |
| kNN | Distance-based baseline |

Scaling is applied inside pipelines for models that need it.

In [ ]:
# ============================================================
# Chapter 9 — Model definitions and evaluation helpers
# ============================================================

def rmse_score(y_true, y_pred):
    # Root mean squared error.
    return mean_squared_error(y_true, y_pred) ** 0.5


def get_regression_models():
    # Return candidate descriptor-based QSAR regression models.
    return {
        "Linear Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]),
        "Ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        ]),
        "Lasso": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.01, max_iter=10000, random_state=RANDOM_STATE)),
        ]),
        "ElasticNet": Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=RANDOM_STATE)),
        ]),
        "Random Forest": RandomForestRegressor(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            random_state=RANDOM_STATE,
        ),
        "SVR RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf", C=10.0, gamma="scale")),
        ]),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=5)),
        ]),
    }


def evaluate_train_test(model, X_train, y_train, X_test, y_test):
    # Fit a model and calculate train/test regression metrics.
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    return {
        "train_MAE": mean_absolute_error(y_train, train_pred),
        "train_RMSE": rmse_score(y_train, train_pred),
        "train_R2": r2_score(y_train, train_pred),
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": rmse_score(y_test, test_pred),
        "test_R2": r2_score(y_test, test_pred),
    }, test_pred

In [ ]:
# Benchmark all models.
models = get_regression_models()

results = []
test_predictions = {}
fitted_models = {}

for model_name, model in models.items():
    metrics, y_pred = evaluate_train_test(model, X_train, y_train, X_test, y_test)
    metrics["model"] = model_name
    results.append(metrics)
    test_predictions[model_name] = y_pred
    fitted_models[model_name] = model

results_df = pd.DataFrame(results)
results_df = results_df[
    [
        "model",
        "train_MAE", "train_RMSE", "train_R2",
        "test_MAE", "test_RMSE", "test_R2",
    ]
].sort_values("test_RMSE").reset_index(drop=True)

display(results_df)

In [ ]:
# Model comparison plot.
plt.figure(figsize=(8, 5))
plt.barh(results_df["model"][::-1], results_df["test_RMSE"][::-1])
plt.xlabel("Test RMSE")
plt.ylabel("Model")
plt.title("Descriptor QSAR model comparison")
plt.show()

# Chapter 10 — Cross-validation

A single train/test split can be noisy. Cross-validation estimates model stability across multiple training subsets.

Here we use 5-fold KFold cross-validation on the training set only.

In [ ]:
# ============================================================
# Chapter 10 — Cross-validation
# ============================================================

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []

for model_name, model in get_regression_models().items():
    neg_mse_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )

    rmse_scores = np.sqrt(-neg_mse_scores)

    cv_rows.append({
        "model": model_name,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std(),
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_RMSE_mean").reset_index(drop=True)
display(cv_results_df)

In [ ]:
# Combine test metrics with cross-validation results.
summary_df = results_df.merge(cv_results_df, on="model", how="left")
summary_df = summary_df.sort_values("CV_RMSE_mean").reset_index(drop=True)

display(summary_df)

# Chapter 11 — Final model diagnostics

The final model is selected by the lowest cross-validation RMSE. We then refit it on the training set and inspect predictions on the held-out test set.

In [ ]:
# ============================================================
# Chapter 11 — Select and refit final model
# ============================================================

best_model_name = cv_results_df.iloc[0]["model"]
final_model = get_regression_models()[best_model_name]

final_model.fit(X_train, y_train)
final_test_pred = final_model.predict(X_test)

final_metrics = {
    "model": best_model_name,
    "test_MAE": mean_absolute_error(y_test, final_test_pred),
    "test_RMSE": rmse_score(y_test, final_test_pred),
    "test_R2": r2_score(y_test, final_test_pred),
}

print("Selected final model:", best_model_name)
display(pd.DataFrame([final_metrics]))

In [ ]:
# Predicted vs actual plot.
plt.figure(figsize=(6, 6))
plt.scatter(y_test, final_test_pred, alpha=0.7)

low = min(float(y_test.min()), float(final_test_pred.min()))
high = max(float(y_test.max()), float(final_test_pred.max()))
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual logS")
plt.ylabel("Predicted logS")
plt.title(f"Predicted vs actual — {best_model_name}")
plt.show()

In [ ]:
# Residual plot.
residuals = y_test.to_numpy() - final_test_pred

plt.figure(figsize=(7, 5))
plt.scatter(final_test_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted logS")
plt.ylabel("Residual: actual - predicted")
plt.title(f"Residual plot — {best_model_name}")
plt.show()

print("Residual summary:")
display(pd.Series(residuals).describe())

## Diagnostic interpretation guide

| Pattern | Interpretation |
|---|---|
| Points close to diagonal | Better prediction accuracy |
| Wide scatter around diagonal | Large prediction error |
| Residuals centred around 0 | Low systematic bias |
| Curved residual pattern | Model may miss nonlinear chemistry |
| Very low train error and high test error | Overfitting |

# Chapter 12 — Molecule-level error analysis

This step maps model errors back to molecules.

This is important for chemistry ML because the model may perform poorly for particular structural classes.

In [ ]:
# ============================================================
# Chapter 12 — Molecule-level error analysis
# ============================================================

error_df = model_df.loc[idx_test, ["name", "canonical_smiles", "target_logS"]].copy()
error_df["predicted_logS"] = final_test_pred
error_df["error"] = error_df["target_logS"] - error_df["predicted_logS"]
error_df["absolute_error"] = error_df["error"].abs()

error_df = error_df.merge(
    model_df[["name", "canonical_smiles"] + selected_feature_names],
    on=["name", "canonical_smiles"],
    how="left"
)

error_df = error_df.sort_values("absolute_error", ascending=False).reset_index(drop=True)

display(error_df.head(15))

In [ ]:
# Draw largest-error molecules for visual inspection.
top_n = min(8, len(error_df))
top_error_smiles = error_df.head(top_n)["canonical_smiles"].tolist()
top_error_names = [
    f"{row.name}: err={row.absolute_error:.2f}"
    for row in error_df.head(top_n).itertuples(index=False)
]

top_error_mols = [Chem.MolFromSmiles(smi) for smi in top_error_smiles]

img = Draw.MolsToGridImage(
    top_error_mols,
    molsPerRow=4,
    subImgSize=(250, 200),
    legends=top_error_names,
)

display(img)

# Chapter 13 — Feature interpretation

Descriptor models are useful because they remain chemically interpretable.

Tree models provide `feature_importances_`. Linear models provide coefficients. For other models, permutation importance can be used.

In [ ]:
# ============================================================
# Chapter 13 — Feature interpretation
# ============================================================

def extract_final_estimator(model):
    # Return the last estimator if the model is a pipeline.
    if hasattr(model, "named_steps") and "model" in model.named_steps:
        return model.named_steps["model"]
    return model


def show_feature_interpretation(model, X_test, y_test, feature_names, top_n=15):
    # Display model-specific feature importance or permutation importance.
    estimator = extract_final_estimator(model)

    if hasattr(estimator, "feature_importances_"):
        importance = estimator.feature_importances_
        importance_type = "model_feature_importance"
    elif hasattr(estimator, "coef_"):
        importance = np.ravel(estimator.coef_)
        importance_type = "linear_coefficient"
    else:
        perm = permutation_importance(
            model,
            X_test,
            y_test,
            n_repeats=10,
            random_state=RANDOM_STATE,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
        )
        importance = perm.importances_mean
        importance_type = "permutation_importance"

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance,
        "absolute_importance": np.abs(importance),
    }).sort_values("absolute_importance", ascending=False)

    print("Importance type:", importance_type)
    display(importance_df.head(top_n))

    plot_df = importance_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["feature"], plot_df["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top descriptor signals — {best_model_name}")
    plt.show()

    return importance_df


importance_df = show_feature_interpretation(
    final_model,
    X_test,
    y_test,
    selected_feature_names,
    top_n=15,
)

## Chemical interpretation checklist

When reviewing feature importance, ask:

- Is `MolLogP` important? This is often expected for solubility because hydrophobic molecules tend to dissolve poorly in water.
- Is `MolWt` or `HeavyAtomCount` important? Larger molecules often have lower solubility.
- Is `TPSA`, `HBD` or `HBA` important? Polar and hydrogen-bonding features often affect aqueous solubility.
- Are ring descriptors important? Rings and aromaticity can reflect hydrophobic scaffolds.
- Does the result make chemical sense, or is the model exploiting dataset-specific shortcuts?

# Chapter 14 — Predict new SMILES

This section turns the notebook into a reusable molecular property predictor.

The function:

1. Parses new SMILES.
2. Calculates the same descriptors.
3. Applies the same feature selection.
4. Predicts logS using the final model.
5. Returns a clean report table.

In [ ]:
# ============================================================
# Chapter 14 — Predict new SMILES
# ============================================================

def featurise_smiles_for_project2(smiles_list, names=None):
    # Create the same descriptor matrix for a list of new SMILES.
    if names is None:
        names = [f"molecule_{i+1}" for i in range(len(smiles_list))]

    rows = []

    for name, smiles in zip(names, smiles_list):
        mol = smiles_to_mol(smiles)

        if mol is None:
            rows.append({
                "name": name,
                "smiles": smiles,
                "valid": False,
                "canonical_smiles": None,
            })
            continue

        desc = calculate_descriptors(mol)
        desc["LipinskiScore"] = lipinski_score(desc)

        rows.append({
            "name": name,
            "smiles": smiles,
            "valid": True,
            "canonical_smiles": mol_to_canonical_smiles(mol),
            **desc,
        })

    feature_df = pd.DataFrame(rows)
    return feature_df


def predict_new_smiles(smiles_list, names=None, model=None):
    # Predict logS for new SMILES using the final Project 2 model.
    if model is None:
        model = final_model

    feature_df = featurise_smiles_for_project2(smiles_list, names=names)
    valid_df = feature_df[feature_df["valid"]].copy()

    if valid_df.empty:
        return feature_df.assign(predicted_logS=np.nan)

    X_new_raw = valid_df[descriptor_cols].copy()
    X_new_raw = X_new_raw.replace([np.inf, -np.inf], np.nan)

    if X_new_raw.isna().any().any():
        raise ValueError("Missing descriptor values found in new molecules.")

    X_new = X_new_raw[selected_feature_names]
    valid_df["predicted_logS"] = model.predict(X_new)

    report_cols = [
        "name",
        "smiles",
        "canonical_smiles",
        "predicted_logS",
        "MolWt",
        "MolLogP",
        "TPSA",
        "HBD",
        "HBA",
        "LipinskiScore",
    ]

    return valid_df[report_cols].sort_values("predicted_logS", ascending=False).reset_index(drop=True)


new_smiles = [
    "CC(=O)OCC1=CC=CC=C1",
    "C(C1C(C(C(C(O1)O)O)O)O)O",
    "CC1(C)CCCC2(C)C1CCC(O2)C(C)(C)O",
    "COC1=C(C=CC(=C1)C=O)O",
]

new_names = ["benzyl acetate", "glucose", "ambroxide-like", "vanillin"]

new_predictions = predict_new_smiles(new_smiles, names=new_names)
display(new_predictions)

In [ ]:
# Draw predicted molecules.
new_mols = [Chem.MolFromSmiles(smi) for smi in new_predictions["canonical_smiles"]]
new_legends = [
    f"{row.name}\nlogS={row.predicted_logS:.2f}"
    for row in new_predictions.itertuples(index=False)
]

img = Draw.MolsToGridImage(
    new_mols,
    molsPerRow=4,
    subImgSize=(260, 220),
    legends=new_legends,
)

display(img)

# Chapter 15 — Single-molecule report helper

This helper returns a compact report for any single molecule.

It is useful for GitHub demos and quick cheminformatics inspection.

In [ ]:
# ============================================================
# Chapter 15 — Single-molecule report helper
# ============================================================

def molecule_qsar_report(smiles, name="query_molecule", show_structure=True):
    # Generate a descriptor and QSAR prediction report for one molecule.
    report = predict_new_smiles([smiles], names=[name])

    if report.empty:
        print("Invalid SMILES.")
        return None

    display(report)

    if show_structure:
        mol = Chem.MolFromSmiles(report.loc[0, "canonical_smiles"])
        display(Draw.MolToImage(mol, size=(280, 220)))

    return report


molecule_qsar_report("CCOC(=O)C", name="ethyl acetate")

# Chapter 16 — Bonus: simple miscibility heuristic

This is not a thermodynamic model. It is a chemistry-inspired descriptor heuristic.

Idea:

- Similar `MolLogP` suggests similar hydrophobicity.
- Similar `TPSA` suggests similar polarity.
- Similar hydrogen-bonding profiles suggest better compatibility.
- Large descriptor differences suggest poorer miscibility.

Use this only as a practice extension, not as a scientific prediction engine.

In [ ]:
# ============================================================
# Chapter 16 — Bonus miscibility heuristic
# ============================================================

def analyse_mixture_smiles(smiles_list, names=None):
    # Calculate descriptors for a mixture candidate list.
    if names is None:
        names = [f"component_{i+1}" for i in range(len(smiles_list))]

    mixture_df = featurise_smiles_for_project2(smiles_list, names=names)
    mixture_df = mixture_df[mixture_df["valid"]].copy()

    keep_cols = [
        "name",
        "smiles",
        "canonical_smiles",
        "MolWt",
        "MolLogP",
        "TPSA",
        "HBD",
        "HBA",
        "RotatableBonds",
    ]

    return mixture_df[keep_cols].reset_index(drop=True)


def predict_miscibility_from_descriptors(mixture_df):
    # Score rough mixture compatibility from descriptor spread.
    if len(mixture_df) < 2:
        return {
            "miscibility_score": np.nan,
            "prediction": "Need at least two valid molecules",
        }

    logp_range = mixture_df["MolLogP"].max() - mixture_df["MolLogP"].min()
    tpsa_range = mixture_df["TPSA"].max() - mixture_df["TPSA"].min()
    hbond_range = (mixture_df["HBD"] + mixture_df["HBA"]).max() - (mixture_df["HBD"] + mixture_df["HBA"]).min()

    score = 100
    score -= min(logp_range * 15, 45)
    score -= min(tpsa_range * 0.4, 35)
    score -= min(hbond_range * 5, 20)
    score = max(0, min(100, score))

    if score >= 70:
        prediction = "likely compatible"
    elif score >= 40:
        prediction = "partly compatible or formulation-dependent"
    else:
        prediction = "likely poorly compatible"

    return {
        "miscibility_score": round(score, 1),
        "prediction": prediction,
        "logp_range": round(logp_range, 2),
        "tpsa_range": round(tpsa_range, 2),
        "hbond_range": round(hbond_range, 2),
    }


def predict_mixture_miscibility(smiles_list, names=None, show_molecules=True):
    # Return a descriptor-based rough miscibility report.
    mixture_df = analyse_mixture_smiles(smiles_list, names=names)

    if mixture_df.empty:
        print("No valid molecules.")
        return None, None

    score_report = predict_miscibility_from_descriptors(mixture_df)

    display(mixture_df)
    display(pd.DataFrame([score_report]))

    if show_molecules:
        mols = [Chem.MolFromSmiles(smi) for smi in mixture_df["canonical_smiles"]]
        legends = mixture_df["name"].tolist()
        display(Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(240, 200), legends=legends))

    return mixture_df, score_report


predict_mixture_miscibility(
    smiles_list=["CO", "CCO", "CCCO"],
    names=["methanol", "ethanol", "propanol"],
)

predict_mixture_miscibility(
    smiles_list=["O", "c1ccccc1"],
    names=["water", "benzene"],
)

# Chapter 17 — Save outputs

This section saves useful artefacts for later projects.

In [ ]:
# ============================================================
# Chapter 17 — Save outputs
# ============================================================

output_descriptor_path = OUTPUT_DIR / "project2_descriptor_dataset.csv"
output_results_path = OUTPUT_DIR / "project2_model_results.csv"
output_cv_path = OUTPUT_DIR / "project2_cv_results.csv"
output_errors_path = OUTPUT_DIR / "project2_error_analysis.csv"
output_predictions_path = OUTPUT_DIR / "project2_new_predictions.csv"

df_desc.drop(columns=["mol"], errors="ignore").to_csv(output_descriptor_path, index=False)
results_df.to_csv(output_results_path, index=False)
cv_results_df.to_csv(output_cv_path, index=False)
error_df.to_csv(output_errors_path, index=False)
new_predictions.to_csv(output_predictions_path, index=False)

print("Saved:")
print(output_descriptor_path)
print(output_results_path)
print(output_cv_path)
print(output_errors_path)
print(output_predictions_path)

# Project 2 summary

This notebook built a full descriptor-based QSAR regression workflow:

```text
SMILES → RDKit Mol → descriptors → descriptor dataset → ML models → logS prediction
```

Key outputs:

- Cleaned molecule dataset
- RDKit descriptor table
- Model comparison table
- Cross-validation table
- Predicted vs actual plot
- Residual plot
- Molecule-level error table
- Feature interpretation
- New SMILES prediction helper
- Bonus miscibility heuristic

This project is the descriptor-based foundation for later fingerprint models, molecular classification and deep-learning chemistry notebooks.